# Step10a (Aluvión) — Mapas Estáticos / Atlas (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 10a — versión independiente para el peligro Aluvión, primera
de 4 tras la decisión del equipo (2026-08-17) de separar el análisis por
tipo de amenaza. Mismo método exacto que `Step10a_Ecuador_Static_Maps.ipynb`
(el original, vista `TODOS`) — la única diferencia es la fuente de datos:
aquí se usan `hci_composite_canton` filtrado a `hazard_type == "debris_flow"`
(con los 132 cantones sin historial de aluvión rellenados a 0/"Muy Baja",
misma decisión confirmada por el equipo que en `Step9_Ecuador_Spatial_Analysis_DebrisFlow.ipynb`)
y `hotspots_gi_star_canton_debris_flow.csv` (Fase 9 debris_flow, ya ejecutada).

**No modifica** el notebook original ni sus salidas — exporta a archivos
nuevos con sufijo `_debris_flow` (`mapa_master_canton_debris_flow.*`,
`outputs/maps/*_debris_flow.png`, `Atlas_HCI_Ecuador_DebrisFlow.pdf`).
**Fecha:** 2026-08-17

**English:** Static maps / atlas for Ecuador, 2015–2025. Builds the master canton dataset that feeds the rest of the product steps (interactive maps, dashboard, profiles) and the static thematic maps: event frequency, severity, ENSO frequency, affected population, agricultural losses, housing losses, chronicity index and hotspots, as individual PNGs and as one consolidated PDF atlas. Variants exist per hazard type (flood, debris flow, landslide, scour).

## 0. Configuración del entorno

**English:** Environment setup: imports, configuration, logger and output folders.

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper that creates folders when missing
from src.utilidades.io_utils import ensure_dir

# Tabular data
import pandas as pd
# Numerical arrays
import numpy as np
# Vector geodata
import geopandas as gpd
# Plotting library
import matplotlib.pyplot as plt
# Colour utilities
import matplotlib.colors as mcolors
# Multi-page PDF writer for the atlas
from matplotlib.backends.backend_pdf import PdfPages
# Classification schemes (Natural Breaks / Jenks)
import mapclassify
# Own Jenks classification for this hazard (the composite step only computed it for the combined view)
import jenkspy  # clasificacion Jenks propia para debris_flow (Fase 8 solo la calculo para hazard_type==TODOS)

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration
config = load_config(CONFIG_PATH)
# Create the notebook logger
logger = get_logger("step10a_static_maps_debris_flow", config, log_dir=PROJECT_ROOT / "logs")
# Log the start of the notebook
logger.info("Fase 10a (debris_flow) iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Output folder for the PNG maps
maps_dir = ensure_dir(PROJECT_ROOT / "outputs" / "maps")
# Output folder for figures
figures_dir = ensure_dir(PROJECT_ROOT / "outputs" / "figures")

# Default font size for all figures
plt.rcParams["font.size"] = 9

## 1. Dataset maestro cantonal

Une, en una sola tabla espacial, todas las métricas ya calculadas en
fases anteriores que se van a mapear — no se recalcula ningún índice,
solo se reutilizan las salidas ya validadas de las Fases 3-9. Se exporta
una vez (`mapa_master_canton.gpkg`) para que el resto de la Fase 10
(mapas interactivos, dashboard, perfiles) lo reutilice sin repetir esta
unión.

**English:** Master canton dataset: joins in one spatial table every metric already computed in earlier steps that will be mapped. No index is recomputed; validated outputs are reused. It is exported once so the remaining product notebooks can read it.

In [ ]:
# Read canton polygons
canton = gpd.read_file(processed_dir / "adm2_canton.geojson")
# Canton codes as strings
canton["canton_code"] = canton["canton_code"].astype(str)
HAZARD_TYPE = "debris_flow"

# --- HCI + 4 sub-índices (Fase 8), filtrado a aluvión, con relleno a 0/"Muy Baja" ---
# Misma decisión confirmada por el equipo (2026-08-17, ver Step9_..._DebrisFlow.ipynb):
# un cantón sin historial de aluvión tiene cronicidad de aluvión
# genuinamente nula, no un dato faltante.
# Read the composite index at canton level
hci = pd.read_csv(processed_dir / "hci_composite_canton.csv", dtype={"canton_code": str})
# NOTA: NO se reutiliza "clase_hci" de hci_composite_canton.csv -- Fase 8
# (Step8, Seccion 3) solo calcula la clasificacion Jenks para hazard_type=="TODOS",
# para las demas vistas (incluida "debris_flow") la columna queda NaN sin calcular.
# La clasificacion propia para debris_flow se recalcula mas abajo (Seccion 4 de este notebook).
hci_debris_flow = hci[hci["hazard_type"] == HAZARD_TYPE][
    ["canton_code", "severity_index", "recurrence_index", "persistence_index",
     "enso_magnitude_mean", "hci"]
]
n_con_historial = len(hci_debris_flow)

# --- Frecuencia de eventos (Fase 3), filtrado a aluvión ---
# Read event recurrence per canton
freq = pd.read_csv(processed_dir / "hist_recurrencia_canton.csv", dtype={"canton_code": str})
freq_debris_flow = freq[freq["hazard_type"] == HAZARD_TYPE][["canton_code", "n_eventos", "anios_afectados"]] \
    .rename(columns={"n_eventos": "frecuencia_aluviones"})

# --- Frecuencia ENSO (Fase 3), filtrado a aluvión, mismo criterio ICEN "El Niño" que el original ---
# Read event counts per ENSO category
enso = pd.read_csv(processed_dir / "hist_enso_canton.csv", dtype={"canton_code": str})
# Keep ICEN 'El Niño' months only (same event definition as the recurrence step)
enso_icen = enso[
    (enso["hazard_type"] == HAZARD_TYPE) & (enso["indice_enso"] == "ICEN")
    & (enso["categoria"].str.startswith("El Niño"))
]
# Sum El Niño events per canton
enso_freq = enso_icen.groupby("canton_code")["n_eventos"].sum().reset_index() \
    .rename(columns={"n_eventos": "frecuencia_enso"})

# --- Hotspots Gi* (Fase 9, debris_flow — ya cubre los 221 cantones, incluidos los 132 rellenados) ---
# Read Getis-Ord Gi* hotspot classes
hotspot = pd.read_csv(processed_dir / "hotspots_gi_star_canton_debris_flow.csv", dtype={"canton_code": str})
# Keep canton code and Gi* class
hotspot_hci = hotspot[["canton_code", "gi_clase_hci"]]

# --- Impactos máximos históricos, recalculados solo de eventos de aluvión ---
# Read all georeferenced events
he_todos = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Keep only events of this hazard type
he = he_todos[he_todos["hazard_type"] == HAZARD_TYPE].copy()
# Canton codes as strings
he["canton_code"] = he["canton_code"].astype(str)
# Population affected = direct + indirect
he["poblacion_total"] = he["affected_direct"].fillna(0) + he["affected_indirect"].fillna(0)
# Crops lost = affected + destroyed hectares
he["agricultura_total"] = he["crops_affected_ha"].fillna(0) + he["crops_destroyed_ha"].fillna(0)
# Housing lost = affected + destroyed houses
he["vivienda_total"] = he["houses_affected"].fillna(0) + he["houses_destroyed"].fillna(0)
# --- Educación y Salud (añadido 2026-09-18, Fase 14, ver docs/methodology/14_variables_complementarias.md) ---
# Schools affected + destroyed
he["educacion_total"] = he["schools_affected"].fillna(0) + he["schools_destroyed"].fillna(0)
# Health facilities affected + destroyed
he["salud_total"] = he["health_facilities_affected"].fillna(0) + he["health_facilities_destroyed"].fillna(0)
# Historical maximum of each impact per canton
impact_max = he.groupby("canton_code")[
    ["poblacion_total", "agricultura_total", "vivienda_total", "educacion_total", "salud_total"]
].max().reset_index()

# --- Conectividad y bienes (añadido 2026-09-18, Fase 14): extraídas por separado de
# BD_2015_2025 (no viven en hazard_event_geo), ya agregadas por cantón x amenaza. ---
# Read complementary variables (roads, bridges, assets)
extra = pd.read_csv(processed_dir / "variables_complementarias_canton_por_amenaza.csv")
# Canton codes zero-padded to 4 digits
extra["canton_code"] = extra["canton_code"].astype(str).str.zfill(4)
# Keep only this hazard type
extra = extra[extra["hazard_type"] == "debris_flow"].drop(columns=["hazard_type"])
# Complementary variables to map
extra_cols = ["vias_metros_afectados", "puentes_total", "bienes_publicos_total", "bienes_privados_total",
              "familias_afectadas", "familias_damnificadas", "damnificados_total", "personas_acogientes"]

# Master canton table: geometry + all metrics to map
master = canton.merge(hci_debris_flow, on="canton_code", how="left") \
    .merge(freq_debris_flow, on="canton_code", how="left") \
    .merge(enso_freq, on="canton_code", how="left") \
    .merge(hotspot_hci, on="canton_code", how="left") \
    .merge(impact_max, on="canton_code", how="left") \
    .merge(extra, on="canton_code", how="left")

# Relleno de los cantones sin historial de aluvión: índices numéricos a 0,
# conteos/frecuencias a 0. La clase HCI se calcula después (Sección 4), no se rellena aquí.
# Numeric indices to fill when a canton has no history
indices_numericos = ["severity_index", "recurrence_index", "persistence_index", "enso_magnitude_mean", "hci"]
# Count cantons without history
n_rellenados = master[indices_numericos].isna().any(axis=1).sum()
# Fill missing indices with 0 (no events = no chronicity)
master[indices_numericos] = master[indices_numericos].fillna(0.0)
master["frecuencia_aluviones"] = master["frecuencia_aluviones"].fillna(0)
master["anios_afectados"] = master["anios_afectados"].fillna(0)
master["frecuencia_enso"] = master["frecuencia_enso"].fillna(0)
master[["poblacion_total", "agricultura_total", "vivienda_total", "educacion_total", "salud_total"]] = master[
    ["poblacion_total", "agricultura_total", "vivienda_total", "educacion_total", "salud_total"]
].fillna(0.0)
# Fill missing complementary variables with 0
master[extra_cols] = master[extra_cols].fillna(0.0)  # cantones sin ese tipo de impacto en esta amenaza -> 0
# Check the filled-canton count matches the expectation
assert n_rellenados == 221 - n_con_historial, "el conteo de cantones rellenados no coincide con lo esperado"
logger.info("Relleno con 0/\"Muy Baja\" aplicado a %d cantones sin historial de aluvión", n_rellenados)

cols_no_geom = [c for c in master.columns if c != "geometry"]
assert master[cols_no_geom].isna().sum().sum() == 0, "quedaron valores nulos en el dataset maestro — revisar uniones"
assert len(master) == 221, f"se esperaban 221 cantones, hay {len(master)}"

# Export the master dataset as GeoPackage
master.to_file(processed_dir / "mapa_master_canton_debris_flow.gpkg", driver="GPKG")
# Export the master dataset as CSV
master.drop(columns="geometry").to_csv(processed_dir / "mapa_master_canton_debris_flow.csv", index=False)
logger.info("Dataset maestro cantonal (debris_flow): %d cantones x %d columnas, exportado", len(master), len(master.columns))
master.head(3)

## 2. Función genérica de mapa coroplético

Una sola función parametrizable para las 6 variables continuas (usa
quiebres naturales/Jenks, 5 clases — misma convención de clasificación
que la Fase 8), evita repetir la misma lógica de graficado 6 veces.
Los mapas categóricos (HCI ya clasificado, Hotspots) usan una función
aparte (Sección 4) porque no son variables continuas a re-clasificar.

**English:** Generic choropleth function: one parametrised function for the continuous variables (Natural Breaks / Jenks, 5 classes, same convention as the composite index) to avoid repeating the plotting logic. Categorical maps (classified chronicity, hotspots) are drawn separately.

In [ ]:
# Province boundaries drawn over each map
PROVINCE_BOUNDARY = gpd.read_file(processed_dir / "adm1_province.geojson")

# Generic choropleth map function (Natural Breaks, 5 classes)
def mapa_coropletico(gdf, columna, titulo, archivo, cmap="YlOrRd", k=5, fmt="{:,.0f}"):
    """Genera y guarda un mapa coroplético cantonal con quiebres naturales (Jenks).

    Args:
        gdf: GeoDataFrame maestro cantonal (con geometría y la columna a mapear).
        columna: nombre de la columna numérica a clasificar y colorear.
        titulo: título del mapa.
        archivo: nombre de archivo (sin extensión) para guardar en outputs/maps.
        cmap: mapa de color de matplotlib.
        k: número de clases (Jenks).
        fmt: formato de número para la leyenda.

    Returns:
        La figura de matplotlib generada (para reutilizar en el atlas PDF).
    """
    # New figure, portrait format
    fig, ax = plt.subplots(figsize=(8, 10))
    # Draw the cantons coloured by class
    gdf.plot(
        column=columna, ax=ax, cmap=cmap, scheme="NaturalBreaks", k=k,
        legend=True, edgecolor="white", linewidth=0.2,
        legend_kwds={"title": titulo, "fmt": fmt, "loc": "lower left", "fontsize": 7, "title_fontsize": 8},
        missing_kwds={"color": "lightgrey"},
    )
    # Overlay province outlines
    PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
    # Map title
    ax.set_title(titulo, fontsize=13, fontweight="bold")
    # Hide the axes
    ax.set_axis_off()
    # Small footer text
    ax.annotate(
        "HCI Ecuador · 2015-2025", xy=(0.99, 0.01), xycoords="axes fraction",
        fontsize=6, color="gray", ha="right",
    )
    # Fit the layout
    fig.tight_layout()
    # Save the PNG
    fig.savefig(maps_dir / f"{archivo}.png", dpi=150, bbox_inches="tight")
    # Log the generated map
    logger.info("Mapa estático generado: %s.png", archivo)
    # Return the figure for the PDF atlas
    return fig

## 3. Los 10 mapas continuos (Jenks, 5 clases)

**English:** The continuous maps (Jenks, 5 classes).

In [ ]:
# Continuous maps: (column, title, file name, colour map)
MAPAS_CONTINUOS = [
    ("frecuencia_aluviones", "Frecuencia de Aluviones (Nº de eventos, 2015-2025)", "01_frecuencia_aluviones_debris_flow", "Blues"),
    ("severity_index", "Índice de Severidad — Aluvión", "02_severidad_debris_flow", "OrRd"),
    ("frecuencia_enso", "Frecuencia de aluviones en meses El Niño", "03_frecuencia_enso_debris_flow", "PuBu"),
    ("poblacion_total", "Población Afectada por Aluvión (máximo histórico)", "04_poblacion_afectada_debris_flow", "YlOrRd"),
    ("agricultura_total", "Pérdidas Agrícolas por Aluvión (máximo histórico, ha)", "05_perdidas_agricolas_debris_flow", "YlGn"),
    ("vivienda_total", "Pérdidas de Vivienda por Aluvión (máximo histórico)", "06_perdidas_vivienda_debris_flow", "Oranges"),
    ("educacion_total", "Establecimientos Educativos Afectados por Aluvión (máximo histórico, afectados+destruidos)", "09_educacion_afectada_debris_flow", "BuPu"),
    ("salud_total", "Centros de Salud Afectados por Aluvión (máximo histórico, afectados+destruidos)", "10_salud_afectada_debris_flow", "RdPu"),
    ("vias_metros_afectados", "Vías Afectadas por Aluvión (máximo histórico, metros lineales)", "11_vias_afectadas_debris_flow", "Greys"),
    ("puentes_total", "Puentes Afectados por Aluvión (máximo histórico, afectados+destruidos)", "12_puentes_afectados_debris_flow", "PuRd"),
]

# Figures kept for the atlas
figuras_atlas = []
# Draw one map per variable
for columna, titulo, archivo, cmap in MAPAS_CONTINUOS:
    fig = mapa_coropletico(master, columna, titulo, archivo, cmap=cmap)
    figuras_atlas.append(fig)
    plt.close(fig)  # libera memoria, ya se guardó el PNG y la referencia para el atlas

## 4. Mapa de Cronicidad HCI — clasificación Jenks recalculada para debris_flow

**A diferencia del notebook original** (donde las clases ya venían fijadas
de Fase 8, calculadas sobre la vista `"TODOS"`), aquí la clasificación
Jenks se recalcula directamente sobre la distribución del HCI-debris_flow (221
cantones, incluidos los 132 rellenados con 0) — Fase 8 (`Step8`, Sección 3)
nunca calculó `clase_hci` para ninguna vista distinta de `"TODOS"`, así
que no hay una clasificación previa que reutilizar. Mismo método exacto
(Jenks, 5 clases, `jenkspy`) y mismas etiquetas (`config["hci_composite"]["class_labels"]`).

**English:** Chronicity index map: for hazard-specific notebooks the Jenks classes are recomputed on the hazard's own distribution; in the combined notebook the classes come from the composite index step.

In [ ]:
# Class names from the configuration
CLASS_LABELS = config["hci_composite"]["class_labels"]  # ["Muy Baja", ..., "Muy Alta"]
# Number of classes
N_CLASSES = len(CLASS_LABELS)
# Sequential red palette, 5 classes
CLASS_COLORS = ["#fee5d9", "#fcae91", "#fb6a4a", "#de2d26", "#a50f15"]  # secuencial rojo, 5 clases
# Map each class to its colour
color_map_hci = dict(zip(CLASS_LABELS, CLASS_COLORS))

# Clasificación Jenks propia para debris_flow (Fase 8 solo la calculó para "TODOS") —
# Lower the first break so the minimum falls in the first bin
# mismo método exacto que Step8 Sección 3 (jenkspy, breaks[0] -= 1e-9 para que
# el mínimo no quede excluido del primer bin).
# Jenks natural breaks computed on this hazard's own distribution
breaks = list(jenkspy.jenks_breaks(master["hci"].values, n_classes=N_CLASSES))
breaks[0] -= 1e-9
# Assign each canton to a class
master["clase_hci"] = pd.cut(master["hci"], bins=breaks, labels=CLASS_LABELS, include_lowest=True)
logger.info("Clasificación Jenks (debris_flow) recalculada, quiebres: %s", [round(b, 4) for b in breaks])
print("Distribución de clases HCI (debris_flow):")
print(master["clase_hci"].value_counts().reindex(CLASS_LABELS))

fig, ax = plt.subplots(figsize=(8, 10))
# Draw one class at a time
for clase, color in color_map_hci.items():
    sub = master[master["clase_hci"] == clase]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2, label=clase)
PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title("Cronicidad de Aluviones por El Niño (HCI)", fontsize=12, fontweight="bold")
ax.set_axis_off()
# Legend handles
handles = [plt.Rectangle((0, 0), 1, 1, color=color_map_hci[c]) for c in CLASS_LABELS]
# Legend
ax.legend(handles, CLASS_LABELS, loc="lower left", fontsize=8, title="Clase HCI (debris_flow)", title_fontsize=9)
ax.annotate("HCI Ecuador · 2015-2025", xy=(0.99, 0.01), xycoords="axes fraction", fontsize=6, color="gray", ha="right")
fig.tight_layout()
fig.savefig(maps_dir / "07_hci_cronicidad_debris_flow.png", dpi=150, bbox_inches="tight")
logger.info("Mapa estático generado: 07_hci_cronicidad_debris_flow.png")
figuras_atlas.append(fig)
plt.close(fig)

# Re-exporta el dataset maestro con clase_hci ya incluida (en la Sección 1
# se exportó antes de calcular la clasificación, que depende de esta sección).
master.to_file(processed_dir / "mapa_master_canton_debris_flow.gpkg", driver="GPKG")
master.drop(columns="geometry").to_csv(processed_dir / "mapa_master_canton_debris_flow.csv", index=False)
logger.info("mapa_master_canton_debris_flow re-exportado con clase_hci incluida")

## 5. Mapa de Hotspots (Getis-Ord Gi*, Fase 9)

**English:** Hotspot map (Getis-Ord Gi*): statistically significant clusters of high (hot) and low (cold) chronicity.

In [ ]:
# Hotspot classes from hot to cold
HOTSPOT_ORDER = [
    "Punto Caliente 99%", "Punto Caliente 95%", "Punto Caliente 90%",
    "No significativo",
    "Punto Frío 90%", "Punto Frío 95%", "Punto Frío 99%",
]
# Diverging palette: red = hot, blue = cold
HOTSPOT_COLORS = ["#67000d", "#cb181d", "#fc9272", "#f0f0f0", "#9ecae1", "#3182bd", "#08306b"]
# Map each hotspot class to its colour
color_map_hotspot = dict(zip(HOTSPOT_ORDER, HOTSPOT_COLORS))

fig, ax = plt.subplots(figsize=(8, 10))
# Draw one class at a time
for clase, color in color_map_hotspot.items():
    sub = master[master["gi_clase_hci"] == clase]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
PROVINCE_BOUNDARY.boundary.plot(ax=ax, color="black", linewidth=0.4)
ax.set_title("Hotspots de Cronicidad HCI — Aluvión (Getis-Ord Gi*)", fontsize=12, fontweight="bold")
ax.set_axis_off()
handles = [plt.Rectangle((0, 0), 1, 1, color=color_map_hotspot[c]) for c in HOTSPOT_ORDER]
ax.legend(handles, HOTSPOT_ORDER, loc="lower left", fontsize=7, title="Clasificación Gi*", title_fontsize=8)
ax.annotate("HCI Ecuador · 2015-2025", xy=(0.99, 0.01), xycoords="axes fraction", fontsize=6, color="gray", ha="right")
fig.tight_layout()
fig.savefig(maps_dir / "08_hotspots_gi_star_debris_flow.png", dpi=150, bbox_inches="tight")
logger.info("Mapa estático generado: 08_hotspots_gi_star_debris_flow.png")
figuras_atlas.append(fig)
plt.close(fig)

## 6. Atlas PDF consolidado (los 8 mapas en un solo documento)

**English:** Consolidated PDF atlas: all maps in one document, with a cover page.

In [ ]:
# Output path of the consolidated atlas PDF
atlas_path = PROJECT_ROOT / "outputs" / "reports" / "Atlas_HCI_Ecuador_DebrisFlow.pdf"
# Create the reports folder
ensure_dir(atlas_path.parent)

# Regenera las figuras en memoria (mismo patrón que el notebook original,
# Fase 10a §6). Lista explícita de nombres (no glob) por la misma razón
# documentada allí: outputs/maps/ contiene mapas de otras fases (SAR, y
# ahora también el atlas TODOS original) que un glob amplio incluiría por error.
# Explicit list of map files (not a glob, other steps also write maps)
nombres_atlas = [c[2] for c in MAPAS_CONTINUOS] + ["07_hci_cronicidad_debris_flow", "08_hotspots_gi_star_debris_flow"]
# Full paths of the PNG files
png_files = [maps_dir / f"{n}.png" for n in nombres_atlas]
# Check every PNG exists
assert all(p.exists() for p in png_files), "falta algún PNG del atlas (debris_flow)"
# Check the expected number of maps
assert len(png_files) == 12, f"se esperaban 12 mapas PNG, hay {len(png_files)}"

# Write the atlas, one map per page
with PdfPages(atlas_path) as pdf:
    # Cover page
    # Portada
    fig, ax = plt.subplots(figsize=(8.5, 11))
    ax.axis("off")
    ax.text(0.5, 0.65, "Atlas de Cronicidad de\nAluviones por El Niño", ha="center", fontsize=22, fontweight="bold")
    ax.text(0.5, 0.58, "Peligro: Aluvión", ha="center", fontsize=14, style="italic")
    ax.text(0.5, 0.50, "Ecuador 2015–2025", ha="center", fontsize=16)
    ax.text(0.5, 0.42, "HCI — Hazard Chronicity Index", ha="center", fontsize=12, style="italic")
    ax.text(0.5, 0.15, "Jerson Achicanoy · 2026-08-17", ha="center", fontsize=10)
    # Save the page
    pdf.savefig(fig)
    plt.close(fig)

    # Add each map as a page
    for png in png_files:
        # Read the PNG
        img = plt.imread(png)
        fig, ax = plt.subplots(figsize=(8.5, 11))
        # Show the image on the page
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

logger.info("Atlas PDF consolidado (debris_flow): %s (%d páginas)", atlas_path, len(png_files) + 1)
# Report the atlas path
print(f"Atlas exportado: {atlas_path}")

## 7. Validación

- Los 8 PNG existen en `outputs/maps/`.
- El PDF del atlas tiene 9 páginas (portada + 8 mapas).
- El dataset maestro (`mapa_master_canton.gpkg`) no tiene nulos y cubre
  los 221 cantones — verificado en la Sección 1 antes de exportar.

**English:** Validation: all PNG files exist, the atlas has the expected number of pages and the master dataset has no nulls and covers all 221 cantons.

In [ ]:
assert all((maps_dir / f"{n}.png").exists() for n in nombres_atlas), "falta algun PNG del atlas de Fase 10 (debris_flow)"
# Check the atlas was written
assert atlas_path.exists() and atlas_path.stat().st_size > 0, "el atlas PDF no se genero correctamente"
logger.info("Fase 10a (debris_flow) - validacion OK. Mapas en %s, atlas en %s", maps_dir, atlas_path)
print("Fase 10a (debris_flow) completa.")